In [10]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import zipfile
import xml.etree.ElementTree as ET

# 1. PROJECT PATHS AND SETTINGS

PROJECT_ROOT = Path(
    r"D:\Big Data Programming Project\Final Assignment"
)

SIRIVM_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "sirivm"
)

SIRIVM_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "sirivm"
)

LOG_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "logs"
)

SERVICE_DATES = [
    "2025-12-26",
    "2025-12-27",
    "2025-12-28"
]

OPERATOR_REF = "SCNE"

# A record must have been updated within five minutes
# of the SIRI-VM snapshot.
MAXIMUM_RECORD_AGE_SECONDS = 300

SIRI_NAMESPACE = {
    "siri": "http://www.siri.org.uk/siri"
}

OUTPUT_COLUMNS = [
    "service_date",
    "source_zip",
    "snapshot_time",
    "recorded_at_time",
    "record_age_seconds",
    "line_ref",
    "published_line_name",
    "direction_ref",
    "data_frame_ref",
    "dated_journey_ref",
    "vehicle_journey_ref",
    "origin_aimed_departure_time",
    "origin_ref",
    "origin_name",
    "destination_ref",
    "destination_name",
    "vehicle_ref",
    "block_ref",
    "longitude",
    "latitude",
    "bearing"
]

SIRIVM_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("SIRI-VM input root:", SIRIVM_INPUT_ROOT)
print("SIRI-VM output root:", SIRIVM_OUTPUT_ROOT)

Project root: D:\Big Data Programming Project\Final Assignment
SIRI-VM input root: D:\Big Data Programming Project\Final Assignment\data\raw\sirivm
SIRI-VM output root: D:\Big Data Programming Project\Final Assignment\data\interim\sirivm


In [11]:
# 2. HELPER FUNCTIONS

def get_text(activity, path):
    """
    Safely retrieve text from a SIRI XML element.
    Returns None when the field does not exist.
    """
    value = activity.findtext(
        path,
        default=None,
        namespaces=SIRI_NAMESPACE
    )

    if value is None:
        return None

    value = value.strip()

    return value if value else None


def get_snapshot_time(zip_path):
    """
    Extract the UTC snapshot timestamp from a filename such as:
    sirivm-20251227T082001.zip
    """
    filename_timestamp = zip_path.stem.replace("sirivm-", "")

    return datetime.strptime(
        filename_timestamp,
        "%Y%m%dT%H%M%S"
    ).replace(tzinfo=timezone.utc)


def extract_scne_records_from_zip(
    zip_path,
    service_date,
    maximum_age_seconds=300
):
    """
    Extract fresh, same-date SCNE VehicleActivity records
    from one SIRI-VM ZIP archive.
    """
    snapshot_time = get_snapshot_time(zip_path)
    extracted_records = []

    with zipfile.ZipFile(zip_path, "r") as archive:

        if "siri.xml" not in archive.namelist():
            raise FileNotFoundError(
                f"siri.xml was not found inside {zip_path.name}"
            )

        with archive.open("siri.xml") as xml_file:

            for event, activity in ET.iterparse(
                xml_file,
                events=("end",)
            ):
                if not activity.tag.endswith("VehicleActivity"):
                    continue

                operator_ref = get_text(
                    activity,
                    ".//siri:OperatorRef"
                )

                recorded_at_text = get_text(
                    activity,
                    "siri:RecordedAtTime"
                )

                if operator_ref != OPERATOR_REF:
                    activity.clear()
                    continue

                if recorded_at_text is None:
                    activity.clear()
                    continue

                try:
                    recorded_at_time = datetime.fromisoformat(
                        recorded_at_text
                    )
                except ValueError:
                    activity.clear()
                    continue

                record_age_seconds = (
                    snapshot_time - recorded_at_time
                ).total_seconds()

                # Keep only records belonging to the required date.
                if recorded_at_time.date().isoformat() != service_date:
                    activity.clear()
                    continue

                # Remove future timestamps and stale observations.
                if not (
                    0
                    <= record_age_seconds
                    <= maximum_age_seconds
                ):
                    activity.clear()
                    continue

                record = {
                    "service_date": service_date,
                    "source_zip": zip_path.name,
                    "snapshot_time": snapshot_time.isoformat(),
                    "recorded_at_time": recorded_at_text,
                    "record_age_seconds": round(
                        record_age_seconds,
                        2
                    ),
                    "line_ref": get_text(
                        activity,
                        ".//siri:LineRef"
                    ),
                    "published_line_name": get_text(
                        activity,
                        ".//siri:PublishedLineName"
                    ),
                    "direction_ref": get_text(
                        activity,
                        ".//siri:DirectionRef"
                    ),
                    "data_frame_ref": get_text(
                        activity,
                        ".//siri:DataFrameRef"
                    ),
                    "dated_journey_ref": get_text(
                        activity,
                        ".//siri:DatedVehicleJourneyRef"
                    ),
                    "vehicle_journey_ref": get_text(
                        activity,
                        ".//siri:VehicleJourneyRef"
                    ),
                    "origin_aimed_departure_time": get_text(
                        activity,
                        ".//siri:OriginAimedDepartureTime"
                    ),
                    "origin_ref": get_text(
                        activity,
                        ".//siri:OriginRef"
                    ),
                    "origin_name": get_text(
                        activity,
                        ".//siri:OriginName"
                    ),
                    "destination_ref": get_text(
                        activity,
                        ".//siri:DestinationRef"
                    ),
                    "destination_name": get_text(
                        activity,
                        ".//siri:DestinationName"
                    ),
                    "vehicle_ref": get_text(
                        activity,
                        ".//siri:VehicleRef"
                    ),
                    "block_ref": get_text(
                        activity,
                        ".//siri:BlockRef"
                    ),
                    "longitude": get_text(
                        activity,
                        ".//siri:Longitude"
                    ),
                    "latitude": get_text(
                        activity,
                        ".//siri:Latitude"
                    ),
                    "bearing": get_text(
                        activity,
                        ".//siri:Bearing"
                    )
                }

                extracted_records.append(record)
                activity.clear()

    return extracted_records

In [12]:
# 3. EXTRACT ALL THREE SERVICE DATES

extraction_summary = []
failed_files = []

for service_date in SERVICE_DATES:

    input_folder = SIRIVM_INPUT_ROOT / service_date
    output_folder = SIRIVM_OUTPUT_ROOT / service_date
    output_folder.mkdir(parents=True, exist_ok=True)

    output_file = (
        output_folder
        / "scne_fresh_sirivm.csv"
    )

    zip_files = sorted(input_folder.glob("*.zip"))

    if not input_folder.exists():
        print(
            f"\nERROR: Input folder does not exist: "
            f"{input_folder}"
        )
        continue

    if not zip_files:
        print(
            f"\nERROR: No ZIP files found for "
            f"{service_date}"
        )
        continue

    extracted_record_count = 0
    successful_zip_count = 0
    failed_zip_count = 0

    print("Starting extraction for:", service_date)
    print("Input ZIP files:", len(zip_files))
    print("Output file:", output_file)

    with output_file.open(
        mode="w",
        newline="",
        encoding="utf-8"
    ) as csv_file:

        writer = csv.DictWriter(
            csv_file,
            fieldnames=OUTPUT_COLUMNS
        )

        writer.writeheader()

        for file_number, zip_path in enumerate(
            zip_files,
            start=1
        ):
            try:
                records = extract_scne_records_from_zip(
                    zip_path=zip_path,
                    service_date=service_date,
                    maximum_age_seconds=(
                        MAXIMUM_RECORD_AGE_SECONDS
                    )
                )

                if records:
                    writer.writerows(records)
                    extracted_record_count += len(records)

                successful_zip_count += 1

            except Exception as error:
                failed_zip_count += 1

                failed_files.append({
                    "service_date": service_date,
                    "zip_file": zip_path.name,
                    "error": str(error)
                })

            if (
                file_number % 500 == 0
                or file_number == len(zip_files)
            ):
                print(
                    f"{service_date} | "
                    f"Processed {file_number:,}/"
                    f"{len(zip_files):,} ZIP files | "
                    f"Records written: "
                    f"{extracted_record_count:,}"
                )

    extraction_summary.append({
        "service_date": service_date,
        "input_zip_files": len(zip_files),
        "successful_zip_files": successful_zip_count,
        "failed_zip_files": failed_zip_count,
        "extracted_records": extracted_record_count,
        "output_file": str(output_file)
    })

    print("\nCompleted:", service_date)
    print("Successful ZIP files:", successful_zip_count)
    print("Failed ZIP files:", failed_zip_count)
    print("Extracted records:", extracted_record_count)
    print("Saved file:", output_file)

Starting extraction for: 2025-12-26
Input ZIP files: 2859
Output file: D:\Big Data Programming Project\Final Assignment\data\interim\sirivm\2025-12-26\scne_fresh_sirivm.csv
2025-12-26 | Processed 500/2,859 ZIP files | Records written: 0
2025-12-26 | Processed 1,000/2,859 ZIP files | Records written: 43
2025-12-26 | Processed 1,500/2,859 ZIP files | Records written: 27,935
2025-12-26 | Processed 2,000/2,859 ZIP files | Records written: 70,054
2025-12-26 | Processed 2,500/2,859 ZIP files | Records written: 85,382
2025-12-26 | Processed 2,859/2,859 ZIP files | Records written: 85,589

Completed: 2025-12-26
Successful ZIP files: 2857
Failed ZIP files: 2
Extracted records: 85589
Saved file: D:\Big Data Programming Project\Final Assignment\data\interim\sirivm\2025-12-26\scne_fresh_sirivm.csv
Starting extraction for: 2025-12-27
Input ZIP files: 2872
Output file: D:\Big Data Programming Project\Final Assignment\data\interim\sirivm\2025-12-27\scne_fresh_sirivm.csv
2025-12-27 | Processed 500/2,8

In [13]:
# 4. SAVE FAILED-FILE LOG

failed_log_file = (
    LOG_OUTPUT_ROOT
    / "sirivm_extraction_failures.csv"
)

with failed_log_file.open(
    mode="w",
    newline="",
    encoding="utf-8"
) as log_file:

    log_columns = [
        "service_date",
        "zip_file",
        "error"
    ]

    writer = csv.DictWriter(
        log_file,
        fieldnames=log_columns
    )

    writer.writeheader()

    if failed_files:
        writer.writerows(failed_files)

print("\nFailure log saved:", failed_log_file)


Failure log saved: D:\Big Data Programming Project\Final Assignment\outputs\logs\sirivm_extraction_failures.csv


In [14]:
# 5. VERIFY THE SAVED OUTPUT FILES

print("FINAL OUTPUT VERIFICATION")

total_extracted_records = 0

for summary in extraction_summary:

    service_date = summary["service_date"]
    output_file = Path(summary["output_file"])

    saved_row_count = 0
    missing_recorded_time = 0
    missing_coordinates = 0
    invalid_service_dates = 0
    maximum_record_age = None

    seen_rows = set()
    exact_duplicate_count = 0

    with output_file.open(
        mode="r",
        newline="",
        encoding="utf-8"
    ) as csv_file:

        reader = csv.DictReader(csv_file)

        for row in reader:
            saved_row_count += 1

            if not row["recorded_at_time"]:
                missing_recorded_time += 1

            if not row["latitude"] or not row["longitude"]:
                missing_coordinates += 1

            if row["service_date"] != service_date:
                invalid_service_dates += 1

            try:
                record_age = float(
                    row["record_age_seconds"]
                )

                if (
                    maximum_record_age is None
                    or record_age > maximum_record_age
                ):
                    maximum_record_age = record_age

            except (TypeError, ValueError):
                pass

            duplicate_key = tuple(
                row[column]
                for column in OUTPUT_COLUMNS
            )

            if duplicate_key in seen_rows:
                exact_duplicate_count += 1
            else:
                seen_rows.add(duplicate_key)

    total_extracted_records += saved_row_count

    print(f"\nDate: {service_date}")
    print("Saved rows:", f"{saved_row_count:,}")
    print(
        "Matches extraction count:",
        saved_row_count == summary["extracted_records"]
    )
    print(
        "Missing recorded times:",
        missing_recorded_time
    )
    print(
        "Missing coordinates:",
        missing_coordinates
    )
    print(
        "Incorrect service dates:",
        invalid_service_dates
    )
    print(
        "Exact duplicate rows:",
        exact_duplicate_count
    )
    print(
        "Maximum record age:",
        maximum_record_age
    )
    print(
        "Failed ZIP files:",
        summary["failed_zip_files"]
    )

print("\n" + "=" * 70)
print("NOTEBOOK 01 SUMMARY")
print("=" * 70)

for summary in extraction_summary:
    print(
        summary["service_date"],
        "| ZIP files:",
        f"{summary['input_zip_files']:,}",
        "| Extracted records:",
        f"{summary['extracted_records']:,}",
        "| Failed ZIPs:",
        summary["failed_zip_files"]
    )

print(
    "\nTotal fresh SCNE records extracted:",
    f"{total_extracted_records:,}"
)

print(
    "Raw SIRI-VM ZIP files processed:",
    f"{sum(item['input_zip_files'] for item in extraction_summary):,}"
)

FINAL OUTPUT VERIFICATION

Date: 2025-12-26
Saved rows: 85,589
Matches extraction count: True
Missing recorded times: 0
Missing coordinates: 0
Incorrect service dates: 0
Exact duplicate rows: 0
Maximum record age: 300.0
Failed ZIP files: 2

Date: 2025-12-27
Saved rows: 495,255
Matches extraction count: True
Missing recorded times: 0
Missing coordinates: 0
Incorrect service dates: 0
Exact duplicate rows: 0
Maximum record age: 300.0
Failed ZIP files: 1

Date: 2025-12-28
Saved rows: 279,671
Matches extraction count: True
Missing recorded times: 0
Missing coordinates: 0
Incorrect service dates: 0
Exact duplicate rows: 0
Maximum record age: 300.0
Failed ZIP files: 0

NOTEBOOK 01 SUMMARY
2025-12-26 | ZIP files: 2,859 | Extracted records: 85,589 | Failed ZIPs: 2
2025-12-27 | ZIP files: 2,872 | Extracted records: 495,255 | Failed ZIPs: 1
2025-12-28 | ZIP files: 2,864 | Extracted records: 279,671 | Failed ZIPs: 0

Total fresh SCNE records extracted: 860,515
Raw SIRI-VM ZIP files processed: 8,59